In [2]:
library(DEqMS)
library(patchwork)
library(tidyverse)

source("../../evaluation_utils/evaluation/DE_analysis.R")
source("../../evaluation_utils/plots/DE_plots.R")
source("../../evaluation_utils/filtering/filtering_normalization.R")

library(jsonlite)

# check datasets

In [ ]:
study_list = c('PDC000127', 'PXD030344', 'PXD042844')

path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/data/')


for (name in study_list) {
  batch_info = read_tsv(paste0(path_to_reports, name, '/metadata.csv'), show_col_types = FALSE)
  intensities = read_csv(paste0(path_to_reports, name, '/report_filtered.csv'), show_col_types = FALSE)
  
  print(paste0('Processing ', name))
  print(paste0('Number of samples: ', ncol(batch_info), "; Number of proteins: ", nrow(intensities)))

  intensities <- filter_per_center(
    intensities, batch_info,
    'Sample', name, 'Dataset', min_number=2
  )
  intensities <- filter_by_condition(
    intensities, batch_info,
    'Sample', c('Tumor', 'Normal'), 'Condition'
    )




}




ERROR: Error in read_tsv(paste0(path_to_reports, name, "/metadata.csv"), show_col_types = FALSE, : unused argument (sep = ",")


In [7]:
batch_info

Sample,Condition,Dataset
<chr>,<chr>,<chr>
Exp056655,Tumor,PXD042844
Exp056657,Tumor,PXD042844
Exp056659,Tumor,PXD042844
Exp056661,Tumor,PXD042844
Exp056663,Tumor,PXD042844
Exp056665,Tumor,PXD042844
Exp056667,Tumor,PXD042844
Exp056669,Tumor,PXD042844
Exp056671,Tumor,PXD042844


# PG lists
perform the analysis for the PGs that are present in both uniformly and non-uniformlt preprocessed datasets

In [ ]:
labs_list = c('lab_A', 'lab_B', 'lab_C', 'lab_D' , 'lab_E')


# uniformly preprocess data
path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/bacterial_data/balanced/')
unifrom_pg_list = NULL

for (name in labs_list) {
  batch_info = read_tsv(paste0(path_to_reports, name, '/metadata_short.tsv'), show_col_types = FALSE)
  intensities = read_tsv(paste0(path_to_reports, name, '/protein_groups_matrix.tsv'), show_col_types = FALSE)
  
  intensities <- filter_per_center(
    intensities %>% column_to_rownames("rowname"), 
    batch_info,
    'file', name, 'lab', min_number=2
  )
  intensities <- filter_by_condition(
    intensities, batch_info,
    'file', c('Glu', 'Pyr'), 'condition'
    )
  intensities <- intensities %>% rownames_to_column('rowname')

  if (is.null(unifrom_pg_list)){ 
    unifrom_pg_list <- intensities$rowname
  } else {unifrom_pg_list <- union(unifrom_pg_list, intensities$rowname)
  }
}
print(paste0('Number of proteins in uniformly preprocessed data: ', length(unifrom_pg_list)))

# non uniform preprocessing
path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/bacterial_data_protocols/')
pg_list = NULL

for (name in labs_list) {
  intensities = read_tsv(paste0(path_to_reports, name, '/protein_groups_matrix.tsv'), show_col_types = FALSE)
  batch_info = read_tsv(paste0(path_to_reports, name, '/metadata_short.tsv'), show_col_types = FALSE)
  
  intensities <- filter_per_center(
    intensities %>% column_to_rownames("rowname"), 
    batch_info,
    'file', name, 'lab', min_number=2
  )
  intensities <- filter_by_condition(
    intensities, batch_info,
    'file', c('Glu', 'Pyr'), 'condition'
    )
  intensities <- intensities %>% rownames_to_column('rowname')
  if (is.null(pg_list)){ 
    pg_list <- intensities$rowname
  } else {pg_list <- union(pg_list, intensities$rowname)
  }
}
print(paste0('Number of proteins in non-uniformly preprocessed data: ', length(pg_list)))

# Intersection of proteins
common_proteins = intersect(pg_list, unifrom_pg_list)
print(paste0('Number of common proteins: ', length(common_proteins)))
# save to file
write.table(common_proteins, file = '/home/yuliya/repos/cosybio/FedProt/data/bacterial_data_protocols/common_proteins.txt', row.names = FALSE, col.names = FALSE)

Filtering by lab  -  2  not-NA per  lab 
	Before filtering: 2549 24 
	After filtering: 2548 24 
Filtering by condition - two not-NA per condition
	Before filtering: 2548 24 
	After filtering: 2517 24 
Filtering by lab  -  2  not-NA per  lab 
	Before filtering: 2846 23 
	After filtering: 2846 23 
Filtering by condition - two not-NA per condition
	Before filtering: 2846 23 
	After filtering: 2824 23 
Filtering by lab  -  2  not-NA per  lab 
	Before filtering: 2820 23 
	After filtering: 2819 23 
Filtering by condition - two not-NA per condition
	Before filtering: 2819 23 
	After filtering: 2765 23 
Filtering by lab  -  2  not-NA per  lab 
	Before filtering: 2813 24 
	After filtering: 2813 24 
Filtering by condition - two not-NA per condition
	Before filtering: 2813 24 
	After filtering: 2780 24 
Filtering by lab  -  2  not-NA per  lab 
	Before filtering: 2401 24 
	After filtering: 2401 24 
Filtering by condition - two not-NA per condition
	Before filtering: 2401 24 
	After filtering: 2361

# Central run

In [4]:
filter_list_META = list()
analyzed_proteins <- list()

In [5]:
labs_list = c('lab_A', 'lab_B', 'lab_C', 'lab_D' , 'lab_E')


path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/bacterial_data/balanced/')

central_intensities = NULL
central_counts = NULL
central_batch_info = NULL

for (name in labs_list) {
    batch_info = read_tsv(paste0(path_to_reports, name, '/metadata_short.tsv'), show_col_types = FALSE)
    intensities = read_tsv(paste0(path_to_reports, name, '/protein_groups_matrix.tsv'), show_col_types = FALSE)
    counts = read_tsv(paste0(path_to_reports, name, '/protein_counts.tsv'), show_col_types = FALSE)

    if(is.null(central_intensities)){
        central_intensities = intensities
        central_counts = counts
        central_batch_info = batch_info
    } else {
        central_intensities = full_join(central_intensities, intensities, by = 'rowname')
        central_counts = full_join(central_counts, counts, by = 'rowname')
        central_batch_info = rbind(central_batch_info, batch_info)
    }
}
central_batch_info <- central_batch_info %>%
    mutate(lab = as.factor(lab), condition = as.factor(condition))

cat('\tNumber of proteins: ', nrow(central_intensities), '\n')
cat('\tNumber of samples: ', ncol(central_intensities)-1, '\n')

central_intensities <- central_intensities %>%
#   filter(rowname %in% fedprot_list) %>%
  column_to_rownames('rowname')
central_counts <- central_counts %>% 
#   filter(rowname %in% fedprot_list) %>%
  column_to_rownames('rowname')
central_intensities <- central_intensities[, central_batch_info$file]

cat('\tNumber of proteins (common filter): ', nrow(central_intensities), '\n')

central_intensities <- filter_na_proteins(central_intensities, central_batch_info, "file")
central_intensities <- filter_by_condition(
    central_intensities, central_batch_info,
    'file', c('Glu', 'Pyr'), 'condition'
    )

# select minimal count across column for each protein (with na.rm = TRUE)
central_counts$count <- apply(central_counts, 1, min, na.rm = TRUE)
central_counts <- central_counts %>% select(count) %>% as.data.frame()

filter_list_META[['Central']] <- rownames(central_intensities)
cat("Rows after all filters:", nrow(central_intensities), "\n")

central_intensities <- log2(central_intensities + 1)

# run DE analysis
design <- make_design(central_batch_info, 'condition', 'lab')
contrasts <- makeContrasts(Glu-Pyr, levels = colnames(design))
de_results <- run_DE(central_intensities, central_counts, design, contrasts)
de_results <- de_results %>% rownames_to_column('Protein')
write.table(
    de_results, 
    file = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/results/central_res.tsv'), 
    sep = "\t", quote = FALSE, row.names = FALSE)

# plot volcano plot
plot_result <- volcano_plot(
    de_results, paste("different preprocessing,", "central", ", Glu/Pyr"),
    pval_threshold = 0.05, logfc_threshold = 0.5,
    show_names = FALSE
)
ggsave(
    file = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/central_volcano_plot.svg'), 
    plot = plot_result, width = 8, height = 5)



	Number of proteins:  3059 
	Number of samples:  118 
	Number of proteins (common filter):  3059 
Filtering out features that have NAs in all columns
	Before filtering: 3059 98 
	After filtering: 3059 98 
Filtering by condition - two not-NA per condition
	Before filtering: 3059 98 
	After filtering: 2863 98 
Rows after all filters: 2863 


Warning message:
“Partial NA coefficients for 591 probe(s)”


## non-uniform preprocessing central run

In [6]:
labs_list = c('lab_A', 'lab_B', 'lab_C', 'lab_D' , 'lab_E')


path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/bacterial_data_protocols/')

central_intensities = NULL
central_counts = NULL
central_batch_info = NULL

for (name in labs_list) {
    batch_info = read_tsv(paste0(path_to_reports, name, '/metadata_short.tsv'), show_col_types = FALSE)
    intensities = read_tsv(paste0(path_to_reports, name, '/protein_groups_matrix.tsv'), show_col_types = FALSE)
    counts = read_tsv(paste0(path_to_reports, name, '/protein_counts.tsv'), show_col_types = FALSE)

    if(is.null(central_intensities)){
        central_intensities = intensities
        central_counts = counts
        central_batch_info = batch_info
    } else {
        central_intensities = full_join(central_intensities, intensities, by = 'rowname')
        central_counts = full_join(central_counts, counts, by = 'rowname')
        central_batch_info = rbind(central_batch_info, batch_info)
    }
}
central_batch_info <- central_batch_info %>%
    mutate(lab = as.factor(lab), condition = as.factor(condition))

cat('\tNumber of proteins: ', nrow(central_intensities), '\n')
cat('\tNumber of samples: ', ncol(central_intensities)-1, '\n')

central_intensities <- central_intensities %>%
  column_to_rownames('rowname')
central_counts <- central_counts %>% 
  column_to_rownames('rowname')
central_intensities <- central_intensities[, central_batch_info$file]

cat('\tNumber of proteins (common filter): ', nrow(central_intensities), '\n')

central_intensities <- filter_na_proteins(central_intensities, central_batch_info, "file")
central_intensities <- filter_by_condition(
    central_intensities, central_batch_info,
    'file', c('Glu', 'Pyr'), 'condition'
    )

# select minimal count across column for each protein (with na.rm = TRUE)
central_counts$count <- apply(central_counts, 1, min, na.rm = TRUE)
central_counts <- central_counts %>% select(count) %>% as.data.frame()

filter_list_META[['Central']] <- rownames(central_intensities)
cat("Rows after all filters:", nrow(central_intensities), "\n")

central_intensities <- log2(central_intensities + 1)

# run DE analysis
design <- make_design(central_batch_info, 'condition', 'lab')
contrasts <- makeContrasts(Glu-Pyr, levels = colnames(design))
de_results <- run_DE(central_intensities, central_counts, design, contrasts)
de_results <- de_results %>% rownames_to_column('Protein')
write.table(
    de_results, 
    file = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/results/central_res_NONU.tsv'), 
    sep = "\t", quote = FALSE, row.names = FALSE)

# plot volcano plot
plot_result <- volcano_plot(
    de_results, paste("different preprocessing,", "central", ", Glu/Pyr"),
    pval_threshold = 0.05, logfc_threshold = 0.5,
    show_names = FALSE
)
ggsave(
    file = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/central_volcano_plot_NONU.svg'), 
    plot = plot_result, width = 8, height = 5)



	Number of proteins:  3121 
	Number of samples:  118 
	Number of proteins (common filter):  3121 
Filtering out features that have NAs in all columns
	Before filtering: 3121 98 
	After filtering: 3120 98 
Filtering by condition - two not-NA per condition
	Before filtering: 3120 98 
	After filtering: 2928 98 
Rows after all filters: 2928 


Warning message:
“Partial NA coefficients for 582 probe(s)”


# Separate run for meta

In [7]:
options(warn=-1)
labs_list = c('lab_A', 'lab_B', 'lab_C', 'lab_D' , 'lab_E')

# empty plot
x <- ggplot() + theme_minimal()

plots_list = list()

path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/bacterial_data_protocols/')

for (name in labs_list) {
    output_path = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/')
    cat('\nLab: ', name, "\n")

    batch_info = read_tsv(paste0(path_to_reports, name, '/metadata_short.tsv'), show_col_types = FALSE)
    intensities = read_tsv(paste0(path_to_reports, name, '/protein_groups_matrix.tsv'), show_col_types = FALSE)
    counts = read_tsv(paste0(path_to_reports, name, '/protein_counts.tsv'), show_col_types = FALSE)

    intensities <- intensities %>% 
        # filter(rowname %in% common_proteins) %>%
        column_to_rownames('rowname')
    counts <- counts %>% 
        # filter(rowname %in% common_proteins) %>%
        column_to_rownames('rowname')
    intensities <- intensities[, batch_info$file]

    # create design matrix for FedProt
    rownames(batch_info) <- batch_info$file
    dummy_df <- model.matrix(~condition - 1, batch_info)
    colnames(dummy_df) <- gsub("condition", "", colnames(dummy_df))
    design <- batch_info %>% select(-c("condition", "QC_condition")) %>% cbind(dummy_df)
    write_tsv(design %>% rownames_to_column(), 
                paste0(path_to_reports, name, "/design.tsv"))

    # filter
    intensities <- filter_by_condition(intensities, batch_info, 
        'file', c('Glu', 'Pyr'), 'condition')
    intensities <- filter_na_proteins(intensities, batch_info, "file")

    filter_list_META[[name]] <- rownames(intensities)
    analyzed_proteins[[name]] <- rownames(intensities)

    cat("Rows after all filters:", nrow(intensities), "\n")
    intensities <- log2(intensities + 1)

    # run DE
    design <- make_design(batch_info, 'condition')
    contrasts <- makeContrasts(Glu - Pyr, levels = colnames(design))
    de_results <- run_DE(intensities, counts, design, contrasts)
    de_results <- de_results %>% rownames_to_column('Protein')
    # write.table(de_results, file = paste0(output_path, name, '_res_FULL.tsv'), sep = "\t", quote = FALSE, row.names = FALSE)
    write.table(de_results, file = paste0(output_path, name, '_res.tsv'), sep = "\t", quote = FALSE, row.names = FALSE)

    # plot volcano plots
    if(name == 'lab_E'){
        plot_separate <- volcano_plot(
            de_results, paste("different preprocessing,", name, ", Glu/Pyr"),
            pval_threshold = 0.05, logfc_threshold = 0.5,
            show_names = FALSE
        )
    } else {
        plot_separate <- volcano_plot(
            de_results, paste("different preprocessing,", name, ", Glu/Pyr"),
            pval_threshold = 0.05, logfc_threshold = 0.5,
            show_names = FALSE, show_legend = FALSE
        )
    }
    plots_list[[name]] = plot_separate
}

layout <- (plots_list[['lab_A']] | plots_list[['lab_B']] | plots_list[['lab_C']]) /
        (plots_list[['lab_D']] | plots_list[['lab_E']] | x)
# save plot
ggsave(file = paste0(output_path, "volcano_plots.svg"), plot = layout, width = 15, height = 8)


write_json(filter_list_META, "/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/filter_list_META.json")


Lab:  lab_A 
Filtering by condition - two not-NA per condition
	Before filtering: 2603 20 
	After filtering: 2577 20 
Filtering out features that have NAs in all columns
	Before filtering: 2577 20 
	After filtering: 2577 20 
Rows after all filters: 2577 

Lab:  lab_B 
Filtering by condition - two not-NA per condition
	Before filtering: 2806 19 
	After filtering: 2780 19 
Filtering out features that have NAs in all columns
	Before filtering: 2780 19 
	After filtering: 2780 19 
Rows after all filters: 2780 

Lab:  lab_C 
Filtering by condition - two not-NA per condition
	Before filtering: 2836 19 
	After filtering: 2773 19 
Filtering out features that have NAs in all columns
	Before filtering: 2773 19 
	After filtering: 2773 19 
Rows after all filters: 2773 

Lab:  lab_D 
Filtering by condition - two not-NA per condition
	Before filtering: 2927 20 
	After filtering: 2900 20 
Filtering out features that have NAs in all columns
	Before filtering: 2900 20 
	After filtering: 2900 20 
Rows a

In [8]:
meta_filter <- NULL
cat("\nIntersection length:",  length(meta_filter))
meta_union <- NULL

# prepare filter for meta-analyses
for (name in labs_list) {
if(is.null(meta_filter)){
    meta_filter <- filter_list_META[[name]]
    cat("\nIntersection length:",  length(meta_filter))
    meta_union <- filter_list_META[[name]]
} else {
    meta_filter <- intersect(meta_filter, filter_list_META[[name]])
    cat("\nIntersection length:",  length(meta_filter))
    meta_union <- union(meta_union, filter_list_META[[name]])
}
}

cat("\n\n\tIntersection length:",  length(meta_filter))
cat("\n\tUnion length:",  length(meta_union))
filter_list_META[['Meta']] <- meta_filter



Intersection length: 0
Intersection length: 2577
Intersection length: 2452
Intersection length: 2423
Intersection length: 2418
Intersection length: 2304

	Intersection length: 2304
	Union length: 3097

In [9]:
analysed_proteins <- list()
analysed_proteins_to_file <- list()
analyzed_proteins_to_file <- list()

analyzed_proteins_to_file[['bacterial_data_protocols']] <- analyzed_proteins

analysed_proteins$central <- filter_list_META[['Central']]
analysed_proteins$meta <- filter_list_META[['Meta']]

analysed_proteins_to_file[['bacterial_data_protocols']] <- analysed_proteins

# write to json
write_json(analysed_proteins_to_file, "/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/analysed_proteins.json")
write_json(analyzed_proteins_to_file, "/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/analysed_proteins_LABS.json")

# Meta run

In [10]:
system(paste0("cd /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/"))

system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_MetaDE.R /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/ lab_A lab_B lab_C lab_D lab_E"))
system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_MetaVolcanoR.R /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/ lab_A lab_B lab_C lab_D lab_E"))
system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_RankProd.R /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/ lab_A lab_B lab_C lab_D lab_E"))

# Copy the resulting files to the desired directory
system(paste0("cp /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/MA_* /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/results/"))


# Only DIA-NN centers

In [11]:
labs_list = c('lab_B', 'lab_C', 'lab_D')

path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/bacterial_data/balanced/')

central_intensities = NULL
central_counts = NULL
central_batch_info = NULL

for (name in labs_list) {
    batch_info = read_tsv(paste0(path_to_reports, name, '/metadata_short.tsv'), show_col_types = FALSE)
    intensities = read_tsv(paste0(path_to_reports, name, '/protein_groups_matrix.tsv'), show_col_types = FALSE)
    counts = read_tsv(paste0(path_to_reports, name, '/protein_counts.tsv'), show_col_types = FALSE)

    if(is.null(central_intensities)){
        central_intensities = intensities
        central_counts = counts
        central_batch_info = batch_info
    } else {
        central_intensities = full_join(central_intensities, intensities, by = 'rowname')
        central_counts = full_join(central_counts, counts, by = 'rowname')
        central_batch_info = rbind(central_batch_info, batch_info)
    }
}
central_batch_info <- central_batch_info %>%
    mutate(lab = as.factor(lab), condition = as.factor(condition))

cat('\tNumber of proteins: ', nrow(central_intensities), '\n')
cat('\tNumber of samples: ', ncol(central_intensities)-1, '\n')

central_intensities <- central_intensities %>%
  column_to_rownames('rowname')
central_counts <- central_counts %>% 
  column_to_rownames('rowname')
central_intensities <- central_intensities[, central_batch_info$file]

cat('\tNumber of proteins (common filter): ', nrow(central_intensities), '\n')

central_intensities <- filter_na_proteins(central_intensities, central_batch_info, "file")
central_intensities <- filter_by_condition(
    central_intensities, central_batch_info,
    'file', c('Glu', 'Pyr'), 'condition'
    )

# select minimal count across column for each protein (with na.rm = TRUE)
central_counts$count <- apply(central_counts, 1, min, na.rm = TRUE)
central_counts <- central_counts %>% select(count) %>% as.data.frame()

cat("Rows after all filters:", nrow(central_intensities), "\n")

central_intensities <- log2(central_intensities + 1)

# run DE analysis
design <- make_design(central_batch_info, 'condition', 'lab')
contrasts <- makeContrasts(Glu-Pyr, levels = colnames(design))
de_results <- run_DE(central_intensities, central_counts, design, contrasts)
de_results <- de_results %>% rownames_to_column('Protein')
write.table(
    de_results, 
    file = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/results/BCD/central_res.tsv'), 
    sep = "\t", quote = FALSE, row.names = FALSE)

# plot volcano plot
plot_result <- volcano_plot(
    de_results, paste("different preprocessing,", "central", ", Glu/Pyr"),
    pval_threshold = 0.05, logfc_threshold = 0.5,
    show_names = FALSE
)
ggsave(
    file = paste0('/home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/central_volcano_plot_BCD.svg'), 
    plot = plot_result, width = 8, height = 5)



	Number of proteins:  3024 
	Number of samples:  70 
	Number of proteins (common filter):  3024 
Filtering out features that have NAs in all columns
	Before filtering: 3024 58 
	After filtering: 3024 58 
Filtering by condition - two not-NA per condition
	Before filtering: 3024 58 
	After filtering: 2954 58 
Rows after all filters: 2954 


In [12]:
system(paste0("cd /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/"))
system(paste0("cp /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/lab_B_res.tsv /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/"))
system(paste0("cp /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/lab_C_res.tsv /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/"))
system(paste0("cp /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/lab_D_res.tsv /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/"))

system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_MetaDE.R /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/ lab_B lab_C lab_D"))
system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_MetaVolcanoR.R /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/ lab_B lab_C lab_D"))
system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_RankProd.R /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/ lab_B lab_C lab_D"))

# Copy the resulting files to the desired directory
system(paste0("cp /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/Meta_DE/META_BCD/MA_* /home/yuliya/repos/cosybio/FedProt/evaluation/bacterial_data_protocols/results/BCD/"))


In [13]:
# all versions of all used packages print
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 24.04.1 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya/miniforge3/envs/FedProt/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] jsonlite_1.8.9    foreach_1.5.2     data.table_1.16.4 ggrepel_0.9.6    
 [5] lubridate_1.9.4   forcats_1.0.0     stringr_1.5.1     dplyr_1.1.4      
 [9] purrr_1.0.2       readr_2.1.5       tidyr_1.3.1       tibble_3.2